# Data Helper Functions

This notebook has all the utility functions we use for loading and transforming data across the project. Import it with `%run` at the top of your notebook to get access to everything here.

## Imports

Just the basic libraries we need for data manipulation.

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load Table Function

This function loads a Delta table from Databricks. If it can't find the table and you provide a CSV path as fallback, it'll load from there instead. Really helpful during development.

In [0]:
def load_table(table_name, path_fallback=None):
    """
    Load a Databricks Delta table or fall back to CSV.
    
    Args:
        table_name: Fully qualified table name (e.g., 'workspace.schema.table')
        path_fallback: Optional CSV path if Delta table doesn't exist
    
    Returns:
        pandas DataFrame
    """
    try:
        df = spark.table(table_name).toPandas()
        print(f"  [{table_name}] {df.shape[0]:,} rows x {df.shape[1]} cols")
    except Exception:
        if path_fallback:
            df = pd.read_csv(path_fallback, low_memory=False)
            print(f"  [CSV fallback: {path_fallback}] {df.shape[0]:,} rows x {df.shape[1]} cols")
        else:
            raise
    return df

## Date Parsing Function

Converts columns to datetime format. Handles UK date format by default (day first), which is common in our datasets.

In [0]:
def parse_dates(df, cols, dayfirst=True):
    """
    Convert specified columns to datetime format.
    
    Args:
        df: pandas DataFrame
        cols: List of column names to parse
        dayfirst: If True, interprets dates as day/month/year (UK format)
    
    Returns:
        DataFrame with parsed date columns
    """
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], dayfirst=dayfirst, errors='coerce')
    return df

## Binary Flags Function

Converts Yes/No text columns into 1/0 numeric flags. Makes it easier to use them in models.

In [0]:
def binary_flags(df, cols, yes_val='Yes'):
    """
    Convert Yes/No columns to 1/0 binary flags.
    
    Args:
        df: pandas DataFrame
        cols: List of column names to convert
        yes_val: The value to treat as 1 (default 'Yes')
    
    Returns:
        DataFrame with binary flag columns
    """
    for c in cols:
        if c in df.columns:
            df[c] = (df[c] == yes_val).astype(int)
    return df

## Display Utilities

Some helper functions to make outputs cleaner and easier to read.

In [0]:
def section(title):
    """
    Print a formatted section header.
    """
    print(f"\n{'='*65}\n  {title}\n{'='*65}")

def pct(n, d):
    """
    Calculate percentage safely (handles zero denominator).
    
    Args:
        n: Numerator
        d: Denominator
    
    Returns:
        Percentage rounded to 2 decimal places
    """
    return round(n/d*100, 2) if d else 0

def display_df(df, title=None, n=20):
    """
    Display a DataFrame with an optional title.
    
    Args:
        df: pandas DataFrame to display
        title: Optional title to print before the DataFrame
        n: Number of rows to show (default 20)
    """
    if title: 
        print(f"\n>>> {title}")
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

## Confirmation

Just confirming the functions are loaded.

In [0]:
print("✓ Data helper functions loaded")
print("  Available: load_table, parse_dates, binary_flags, section, pct, display_df")